# 2 - Fixed Stratified 5-Fold Outer Splits - All Families

** Create fixed stratified 5-fold outer splits**


Each original `pair_id` is assigned to exactly one outer test fold.  
The resulting fold assignments are saved and must remain unchanged for the rest of the experiment.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd()
AGG_DIR = PROJECT_ROOT / "data" / "agg"
SPLIT_DIR = PROJECT_ROOT / "data" / "split"
STATS_DIR = PROJECT_ROOT / "Stats"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Input folder :", AGG_DIR)
print("Split folder :", SPLIT_DIR)
print("Stats folder :", STATS_DIR)

assert AGG_DIR.exists(), f"Aggregated-data folder not found: {AGG_DIR}"

Project root : c:\Users\riskf\OneDrive\A-DTI2026
Input folder : c:\Users\riskf\OneDrive\A-DTI2026\data\agg
Split folder : c:\Users\riskf\OneDrive\A-DTI2026\data\split
Stats folder : c:\Users\riskf\OneDrive\A-DTI2026\Stats


## 1. Experimental split configuration



In [2]:
N_OUTER_FOLDS = 5

# ---------------------------------------------------------------------
# REQUIRED: set the fixed integer seed before running this notebook.
# Example syntax only: OUTER_SPLIT_SEED = <integer>
# ---------------------------------------------------------------------
OUTER_SPLIT_SEED = 13571113

assert isinstance(OUTER_SPLIT_SEED, int), (
    "Set OUTER_SPLIT_SEED to the fixed integer seed selected for the experiment "
    "before generating the outer folds."
)

FAMILIES = {
    "enzyme": "Enzyme",
    "gpcr": "GPCR",
    "ion_channel": "Ion Channel",
    "nuclear_receptor": "Nuclear Receptor",
}

print("Outer folds :", N_OUTER_FOLDS)
print("Split seed  :", OUTER_SPLIT_SEED)

Outer folds : 5
Split seed  : 13571113


## 2. Load the four canonical pair datasets

Only `pairs.csv` files from Notebook 1 are used.

Required columns:

- `pair_id`
- `family`
- `compound_id`
- `protein_id`
- `compound_index`
- `protein_index`
- `y`

In [3]:
PAIR_DATA = {}

required_columns = {
    "pair_id",
    "family",
    "compound_id",
    "protein_id",
    "compound_index",
    "protein_index",
    "y",
}

for family, display_name in FAMILIES.items():

    path = AGG_DIR / family / "pairs.csv"

    assert path.exists(), (
        f"{display_name}: missing Notebook 1 output: {path}"
    )

    df = pd.read_csv(path)

    missing_columns = required_columns - set(df.columns)

    assert not missing_columns, (
        f"{display_name}: missing required columns: "
        f"{sorted(missing_columns)}"
    )

    assert df["pair_id"].is_unique, (
        f"{display_name}: pair_id is not unique."
    )

    assert set(df["y"].unique()).issubset({0, 1}), (
        f"{display_name}: y contains values outside {{0,1}}."
    )

    PAIR_DATA[family] = df

    print(
        f"{display_name:<18} "
        f"pairs={len(df):,} | "
        f"positive={(df['y'] == 1).sum():,} | "
        f"non-interaction={(df['y'] == 0).sum():,}"
    )

Enzyme             pairs=295,480 | positive=2,926 | non-interaction=292,554
GPCR               pairs=21,185 | positive=635 | non-interaction=20,550
Ion Channel        pairs=42,840 | positive=1,476 | non-interaction=41,364
Nuclear Receptor   pairs=1,404 | positive=90 | non-interaction=1,314


## 3. Create the fixed stratified 5-fold assignments

For each family, `StratifiedKFold` assigns each original observation to exactly one outer test fold.

The label vector `y` is used only to preserve the class distribution across folds.

The fold number is stored as `1, 2, 3, 4, 5`.

In [4]:
OUTER_ASSIGNMENTS = {}

for family, display_name in FAMILIES.items():

    df = PAIR_DATA[family].copy()

    skf = StratifiedKFold(
        n_splits=N_OUTER_FOLDS,
        shuffle=True,
        random_state=OUTER_SPLIT_SEED,
    )

    # Initialize as missing so incomplete assignment cannot pass silently.
    df["outer_fold"] = pd.NA

    X_dummy = np.zeros((len(df), 1))
    y = df["y"].to_numpy()

    for fold_number, (_, test_idx) in enumerate(
        skf.split(X_dummy, y),
        start=1,
    ):
        df.loc[test_idx, "outer_fold"] = fold_number

    df["outer_fold"] = df["outer_fold"].astype(int)

    OUTER_ASSIGNMENTS[family] = df

    print(
        f"{display_name:<18} "
        f"assigned to {df['outer_fold'].nunique()} folds"
    )

Enzyme             assigned to 5 folds
GPCR               assigned to 5 folds
Ion Channel        assigned to 5 folds
Nuclear Receptor   assigned to 5 folds


## 4. Audit the outer folds



In [5]:
fold_stats_rows = []
family_stats_rows = []

for family, display_name in FAMILIES.items():

    original = PAIR_DATA[family]
    assigned = OUTER_ASSIGNMENTS[family]

    # -----------------------------------------------------------------
    # Assignment integrity
    # -----------------------------------------------------------------
    assert len(assigned) == len(original), (
        f"{display_name}: number of assigned rows differs from original."
    )

    assert assigned["pair_id"].is_unique, (
        f"{display_name}: duplicate pair_id in outer assignments."
    )

    assert not assigned["outer_fold"].isna().any(), (
        f"{display_name}: at least one pair has no outer fold."
    )

    assert set(assigned["outer_fold"].unique()) == set(
        range(1, N_OUTER_FOLDS + 1)
    ), (
        f"{display_name}: outer-fold labels are incomplete."
    )

    assert set(assigned["pair_id"]) == set(original["pair_id"]), (
        f"{display_name}: outer-fold pair IDs do not match the original dataset."
    )

    # -----------------------------------------------------------------
    # Fold-level statistics
    # -----------------------------------------------------------------
    total_from_folds = 0
    positive_from_folds = 0
    negative_from_folds = 0

    for fold in range(1, N_OUTER_FOLDS + 1):

        fold_df = assigned[assigned["outer_fold"] == fold]

        n_total = len(fold_df)
        n_positive = int((fold_df["y"] == 1).sum())
        n_non_interaction = int((fold_df["y"] == 0).sum())

        assert n_positive > 0, (
            f"{display_name}, fold {fold}: no positive observations."
        )

        assert n_non_interaction > 0, (
            f"{display_name}, fold {fold}: no non-interaction observations."
        )

        total_from_folds += n_total
        positive_from_folds += n_positive
        negative_from_folds += n_non_interaction

        fold_stats_rows.append({
            "family": display_name,
            "outer_fold": fold,
            "n_test": n_total,
            "n_positive": n_positive,
            "n_non_interaction": n_non_interaction,
            "positive_rate": n_positive / n_total,
            "non_interaction_rate": n_non_interaction / n_total,
        })

    original_positive = int((original["y"] == 1).sum())
    original_negative = int((original["y"] == 0).sum())

    assert total_from_folds == len(original)
    assert positive_from_folds == original_positive
    assert negative_from_folds == original_negative

    family_stats_rows.append({
        "family": display_name,
        "n_pairs": len(original),
        "n_positive": original_positive,
        "n_non_interaction": original_negative,
        "positive_rate": original_positive / len(original),
        "n_outer_folds": N_OUTER_FOLDS,
        "outer_split_seed": OUTER_SPLIT_SEED,
        "assignment_complete": True,
    })

    print(f"PASS - {display_name}")

PASS - Enzyme
PASS - GPCR
PASS - Ion Channel
PASS - Nuclear Receptor


## 5. Inspect fold statistics

In [11]:
FOLD_STATS = pd.DataFrame(fold_stats_rows)
FAMILY_STATS = pd.DataFrame(family_stats_rows)

display(FOLD_STATS)
RATE_CHECK.to_excel(
    STATS_DIR / "fold_stats_check.xlsx",
    index=False,
    sheet_name="Fold_Stats"
)
print(f"Rate check saved to: {STATS_DIR / 'Fold_Stats.xlsx'}")


display(FAMILY_STATS)

RATE_CHECK.to_excel(
    STATS_DIR / "family_stats_check.xlsx",
    index=False,
    sheet_name="Family_Stats"
)
print(f"Rate check saved to: {STATS_DIR / 'Family_Stats.xlsx'}")


,family,outer_fold,n_test,n_positive,n_non_interaction,positive_rate,non_interaction_rate
0,Enzyme,1,59096,586,58510,0.009916,0.990084
1,Enzyme,2,59096,585,58511,0.009899,0.990101
2,Enzyme,3,59096,585,58511,0.009899,0.990101
3,Enzyme,4,59096,585,58511,0.009899,0.990101
4,Enzyme,5,59096,585,58511,0.009899,0.990101
5,GPCR,1,4237,127,4110,0.029974,0.970026
6,GPCR,2,4237,127,4110,0.029974,0.970026
7,GPCR,3,4237,127,4110,0.029974,0.970026
8,GPCR,4,4237,127,4110,0.029974,0.970026
9,GPCR,5,4237,127,4110,0.029974,0.970026


Rate check saved to: c:\Users\riskf\OneDrive\A-DTI2026\Stats\Fold_Stats.xlsx


,family,n_pairs,n_positive,n_non_interaction,positive_rate,n_outer_folds,outer_split_seed,assignment_complete
0,Enzyme,295480,2926,292554,0.009903,5,13571113,True
1,GPCR,21185,635,20550,0.029974,5,13571113,True
2,Ion Channel,42840,1476,41364,0.034454,5,13571113,True
3,Nuclear Receptor,1404,90,1314,0.064103,5,13571113,True


Rate check saved to: c:\Users\riskf\OneDrive\A-DTI2026\Stats\Family_Stats.xlsx


## 6. Additional stratification check

For each family, compare the positive rate of each outer fold with the positive rate of the complete dataset.

In [10]:
rate_check_rows = []

for family, display_name in FAMILIES.items():

    assigned = OUTER_ASSIGNMENTS[family]

    overall_rate = assigned["y"].mean()

    for fold in range(1, N_OUTER_FOLDS + 1):

        fold_df = assigned[assigned["outer_fold"] == fold]
        fold_rate = fold_df["y"].mean()

        rate_check_rows.append({
            "family": display_name,
            "outer_fold": fold,
            "overall_positive_rate": overall_rate,
            "fold_positive_rate": fold_rate,
            "absolute_difference": abs(fold_rate - overall_rate),
        })

RATE_CHECK = pd.DataFrame(rate_check_rows)
display(RATE_CHECK)

RATE_CHECK.to_excel(
    STATS_DIR / "outer_cv_rate_check.xlsx",
    index=False,
    sheet_name="Rate Check"
)

print(f"Rate check saved to: {STATS_DIR / 'outer_cv_rate_check.xlsx'}")




,family,outer_fold,overall_positive_rate,fold_positive_rate,absolute_difference
0,Enzyme,1,0.009903,0.009916,0.000014
1,Enzyme,2,0.009903,0.009899,0.000003
2,Enzyme,3,0.009903,0.009899,0.000003
3,Enzyme,4,0.009903,0.009899,0.000003
4,Enzyme,5,0.009903,0.009899,0.000003
5,GPCR,1,0.029974,0.029974,0.000000
6,GPCR,2,0.029974,0.029974,0.000000
7,GPCR,3,0.029974,0.029974,0.000000
8,GPCR,4,0.029974,0.029974,0.000000
9,GPCR,5,0.029974,0.029974,0.000000


Rate check saved to: c:\Users\riskf\OneDrive\A-DTI2026\Stats\outer_cv_rate_check.xlsx


## 7. Save the permanent outer-fold mappings

The file contains the original pair information plus `outer_fold`.

These files are the authoritative outer split assignments for all later notebooks.

In [8]:
for family, display_name in FAMILIES.items():

    family_split_dir = SPLIT_DIR / family
    family_split_dir.mkdir(parents=True, exist_ok=True)

    output_file = family_split_dir / "outer_folds.csv"

    OUTER_ASSIGNMENTS[family].to_csv(
        output_file,
        index=False,
    )

    print(f"Saved - {display_name}: {output_file}")

Saved - Enzyme: c:\Users\riskf\OneDrive\A-DTI2026\data\split\enzyme\outer_folds.csv
Saved - GPCR: c:\Users\riskf\OneDrive\A-DTI2026\data\split\gpcr\outer_folds.csv
Saved - Ion Channel: c:\Users\riskf\OneDrive\A-DTI2026\data\split\ion_channel\outer_folds.csv
Saved - Nuclear Receptor: c:\Users\riskf\OneDrive\A-DTI2026\data\split\nuclear_receptor\outer_folds.csv


## 8. Save all split statistics to `Stats/`


In [9]:
protocol_df = pd.DataFrame([
    {
        "parameter": "n_outer_folds",
        "value": N_OUTER_FOLDS,
    },
    {
        "parameter": "outer_split_seed",
        "value": OUTER_SPLIT_SEED,
    },
    {
        "parameter": "split_method",
        "value": "StratifiedKFold",
    },
    {
        "parameter": "shuffle",
        "value": True,
    },
])

stats_file = STATS_DIR / "outer_cv_split_statistics.xlsx"

with pd.ExcelWriter(stats_file, engine="openpyxl") as writer:

    FAMILY_STATS.to_excel(
        writer,
        sheet_name="Family Summary",
        index=False,
    )

    FOLD_STATS.to_excel(
        writer,
        sheet_name="Fold Statistics",
        index=False,
    )

    RATE_CHECK.to_excel(
        writer,
        sheet_name="Rate Check",
        index=False,
    )

    protocol_df.to_excel(
        writer,
        sheet_name="Protocol",
        index=False,
    )

print(f"Statistics saved to: {stats_file}")

Statistics saved to: c:\Users\riskf\OneDrive\A-DTI2026\Stats\outer_cv_split_statistics.xlsx


## 9. Save the split protocol manifest

This is not a statistical table. It records the exact split-generation configuration used by the project.

In [12]:
manifest = {
    "step": "05 - fixed stratified 5-fold outer splits",
    "n_outer_folds": N_OUTER_FOLDS,
    "outer_split_seed": OUTER_SPLIT_SEED,
    "splitter": "sklearn.model_selection.StratifiedKFold",
    "shuffle": True,
    "families": list(FAMILIES.keys()),
    "assignment_files": {
        family: str(
            (SPLIT_DIR / family / "outer_folds.csv").relative_to(PROJECT_ROOT)
        )
        for family in FAMILIES
    },
}

manifest_file = SPLIT_DIR / "outer_split_manifest.json"

with open(manifest_file, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest saved to: {manifest_file}")

Manifest saved to: c:\Users\riskf\OneDrive\A-DTI2026\data\split\outer_split_manifest.json


## 10. Reproducibility check



In [13]:
for family, display_name in FAMILIES.items():

    saved = pd.read_csv(
        SPLIT_DIR / family / "outer_folds.csv"
    )

    original = PAIR_DATA[family].copy()

    regenerated = original.copy()
    regenerated["outer_fold"] = pd.NA

    skf = StratifiedKFold(
        n_splits=N_OUTER_FOLDS,
        shuffle=True,
        random_state=OUTER_SPLIT_SEED,
    )

    X_dummy = np.zeros((len(regenerated), 1))
    y = regenerated["y"].to_numpy()

    for fold_number, (_, test_idx) in enumerate(
        skf.split(X_dummy, y),
        start=1,
    ):
        regenerated.loc[test_idx, "outer_fold"] = fold_number

    regenerated["outer_fold"] = regenerated["outer_fold"].astype(int)

    check = saved[["pair_id", "outer_fold"]].merge(
        regenerated[["pair_id", "outer_fold"]],
        on="pair_id",
        suffixes=("_saved", "_regenerated"),
        validate="one_to_one",
    )

    assert (
        check["outer_fold_saved"]
        == check["outer_fold_regenerated"]
    ).all(), (
        f"{display_name}: regenerated split does not match saved split."
    )

    print(f"REPRODUCIBLE - {display_name}")

REPRODUCIBLE - Enzyme
REPRODUCIBLE - GPCR
REPRODUCIBLE - Ion Channel
REPRODUCIBLE - Nuclear Receptor


## 11. Final integrity check


The next notebook will implement ** fixed inner 80/20 train-validation splits within each outer-training fold**.

In [14]:
for family, display_name in FAMILIES.items():

    split_file = SPLIT_DIR / family / "outer_folds.csv"

    assert split_file.exists(), (
        f"{display_name}: missing outer-fold assignment file."
    )

    df = pd.read_csv(split_file)

    assert df["pair_id"].is_unique
    assert set(df["outer_fold"].unique()) == {1, 2, 3, 4, 5}

    # Every original observation appears once in the saved mapping.
    original = PAIR_DATA[family]

    assert len(df) == len(original)
    assert set(df["pair_id"]) == set(original["pair_id"])

    print(f"PASS - {display_name}")

assert (STATS_DIR / "outer_cv_split_statistics.xlsx").exists()
assert (SPLIT_DIR / "outer_split_manifest.json").exists()

print("\nNotebook 2 completed successfully.")
print("Next: Notebook 3 - fixed inner 80/20 train-validation splits.")

PASS - Enzyme
PASS - GPCR
PASS - Ion Channel
PASS - Nuclear Receptor

Notebook 2 completed successfully.
Next: Notebook 3 - fixed inner 80/20 train-validation splits.
